# Pandoc + DeepSeek-OCR — bản code gốc đã sửa

Chạy **cell code bên dưới một lần để mở bộ chọn đề**. Chọn đúng DOCX trong danh sách rồi bấm **Đọc đề đã chọn**. Không tự chạy đề Toán mẫu.

- File phải đã được thêm vào Input của Kaggle. Bấm **Làm mới danh sách** nếu vừa thêm file. Đổi đề bằng danh sách này, không sửa các chuỗi đường dẫn bên trong code dài.
- Mỗi lần đổi/chạy đề, tham chiếu kết quả cũ được xóa; nếu chạy lỗi thì không mở lại kết quả cũ. Giao diện hiện tên file đang đọc, tên file của kết quả, và kiểm tra DOCX đính kèm kết quả khớp file đã chọn.
- Bật Internet và GPU để chạy DeepSeek-OCR. Lần đầu cài thư viện riêng và tải model; không thay Torch/Transformers của kernel.
- Giữ model DeepSeek-OCR, BF16, eager và các cấu hình 384/448/512/640 theo code gốc. Không hứa thời gian/độ chính xác khi chưa chạy GPU thực tế.
- `OCR_DIAGRAMS=False`: giữ hình sơ đồ, không gửi OCR mặc định như luồng gốc. Heuristic có thể phân loại sai; bật True nếu muốn OCR cả các hình này.
- Nhận diện thêm lưới tọa độ/trục đậm để không OCR nhầm các đồ thị câu 1–2. Danh sách nhãn rời kiểu `y x O -2 -1 ...` không được chèn thành một công thức dưới hình; chữ thô vẫn lưu trong báo cáo. Công thức/ma trận không bị xóa bằng bộ lọc nhãn này.
- `OCR_SMALL_IMAGES=True`: giữ khả năng OCR công thức nhỏ; không phóng mọi ảnh lên 384 px trước khi chọn cấu hình.
- Không sửa nội dung LaTeX/ma trận bằng regex. Không coi alt-text là kết quả model đọc được.
- Bản HTML dựng bảng OCR thành hàng/cột và render công thức `\(...\)`, `\[...\]`. Mặc định in bảng/công thức đã dựng, không in lặp ảnh gốc và ghi chú; bật **Đối chiếu ảnh gốc / ghi chú OCR** để xem nguồn trên màn hình. Những đoạn chưa đủ cấu trúc giữ ảnh gốc, không tự đoán thành bảng/công thức.
- Chỉ kiểm tra các ảnh Pandoc xuất được. WMF/EMF/OLE không đọc được sẽ được báo, không tự cài LibreOffice. Giữ thêm DOCX gốc trong ZIP để kiểm tra hình bị thiếu.

Sau khi chạy xong, **kết quả trích xuất + OCR hiện ngay dưới cell**. Ảnh gốc giữ nguyên, chữ model đọc được hiện dưới ảnh với nhãn `[OCR — cần đối chiếu]`. Đây là đầu ra thật của pipeline, không phải xuất lại DOCX gốc rồi gọi là OCR.

Có nút tải **HTML kết quả**, **Markdown** và **ZIP**. Mở HTML để In/lưu PDF; công thức dùng MathML, không cần tải MathJax. HTML không cam kết giống tuyệt đối ngắt dòng/ngắt trang Word. Báo cáo ghi thời gian từng bước và các lượt thử; “có chữ trả về” không phải tỷ lệ chính xác.

Ngay dưới kết quả có **TỔNG THỜI GIAN ĐÃ CHẠY** (phút/giây), gồm chuẩn bị thư viện, Pandoc/ảnh, nạp model, OCR, xuất HTML và nén nội dung ZIP. Số đo dừng khi ZIP nội dung đóng xong, trước thao tác ghi báo cáo thời gian và dựng màn hình. `metrics.json` trong ZIP chứa cùng số đo, không cần chạy lại OCR để xem.

Khi chưa có log mới, cứ mỗi 5 giây có dòng PID/thời gian để phân biệt tiến trình chưa kết thúc. Không tự chạy lại cell cùng lúc. Mỗi lượt có thư mục riêng, không xóa kết quả cũ.


In [ ]:
# CHẠY CELL ĐỂ MỞ BỘ CHỌN ĐỀ. KHÔNG TỰ CHẠY ĐỀ MẪU.
INPUT_DOCX = ""
OCR_DIAGRAMS = False
OCR_SMALL_IMAGES = True
RUN_ROOT = '/kaggle/working/hybrid_original_fixed'

import importlib.util
from pathlib import Path

SCRIPT_SOURCE = '"""Bản chạy thẳng dựa trên code Pandoc + DeepSeek-OCR người dùng gửi.\n\nKhông dùng UI cũ, không clone repository, không tự cài LibreOffice.\nModel/cấu hình suy luận giữ theo code gốc. Hậu xử lý không viết lại công thức.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nfrom collections import deque\nfrom copy import deepcopy\nfrom datetime import datetime\nimport hashlib\nimport html\nfrom html.parser import HTMLParser\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nimport types\nfrom urllib.parse import unquote, urlsplit\nfrom uuid import uuid4\nimport xml.etree.ElementTree as ET\nimport zipfile\n\n\n# Chỉ sửa đường dẫn này nếu dùng file đề khác.\nINPUT_DOCX = ""  # Chọn DOCX bằng bộ chọn đề hoặc truyền --input; không mặc định đề Toán.\nRUN_ROOT = "/kaggle/working/hybrid_original_fixed"\nOCR_DIAGRAMS = False       # Giống luồng gốc: giữ hình sơ đồ, không OCR mặc định.\nOCR_SMALL_IMAGES = True    # Không bỏ công thức chỉ vì một cạnh nhỏ hơn 80 px.\nMODEL_NAME = "deepseek-ai/DeepSeek-OCR"\nMAX_NEW_TOKENS = 1000\nHEARTBEAT_SECONDS = 5.0\n\n# Cài riêng cho tiến trình con; tuyệt đối không pip install torch/torchvision.\nREQUIREMENTS = [\n    "transformers==4.46.3", "tokenizers==0.20.3", "huggingface-hub==0.36.2",\n    "Pillow>=10.1,<13", "numpy>=1.26,<3", "pymupdf>=1.24,<2",\n    "einops>=0.8", "addict>=2.4", "easydict>=1.13", "accelerate>=0.26,<2",\n    "safetensors>=0.4.3", "filelock>=3.16", "packaging>=23",\n    "pyyaml>=6", "regex>=2024", "requests>=2.32,<3", "tqdm>=4.66",\n    "fsspec>=2023.5", "typing-extensions>=4.9", "hf-xet>=1.1.3,<2",\n    "psutil>=5.9", "urllib3>=1.26,<3", "certifi>=2024",\n    "charset-normalizer>=3,<4", "idna>=3,<4", "pypandoc-binary>=1.14,<2",\n]\nIMAGE_RE = re.compile(r\'!\\[[^\\]\\n]*\\]\\((<[^>]+>|[^)\\n]+)\\)(\\{[^}]*\\})?\')\nGARBAGE_PATTERNS = [r\'"[Ll]ine":\\s*[\\[{]\', r\'Image ID:\\s*[A-Z0-9]+\',\n                    r\'"line_type"\', r\'"line_endpoint"\']\n\n\ndef validate_docx(value):\n    if value is None or not str(value).strip():\n        raise ValueError("Điền đường dẫn đầy đủ vào INPUT_DOCX ở đầu cell.")\n    source = Path(value).expanduser().resolve()\n    if not source.is_file() or source.suffix.lower() != ".docx":\n        raise ValueError("Không phải file DOCX: " + str(source))\n    try:\n        with zipfile.ZipFile(source) as archive:\n            if not {"[Content_Types].xml", "word/document.xml"}.issubset(archive.namelist()):\n                raise ValueError("File không có cấu trúc Word DOCX hợp lệ.")\n    except zipfile.BadZipFile as error:\n        raise ValueError("DOCX bị hỏng hoặc không phải ZIP Word.") from error\n    return source\n\n\ndef stream_command(command, *, log, env=None, cwd=None, step="Tiến trình", run_started=None):\n    """In log trực tiếp và báo PID khi tiến trình chưa có log mới."""\n    print("\\n>>> " + step, flush=True)\n    log.write("\\n>>> " + step + "\\n")\n    log.flush()\n    kwargs = {"creationflags": subprocess.CREATE_NO_WINDOW} if os.name == "nt" else {}\n    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               text=True, encoding="utf-8", errors="replace", bufsize=1,\n                               env=env, cwd=cwd, **kwargs)\n    pending, tail = queue.Queue(), deque(maxlen=45)\n    started = last_output = last_tick = time.monotonic()\n    total_started = time.perf_counter() if run_started is None else run_started\n    def read_output():\n        try:\n            for line in process.stdout:\n                pending.put(("line", line))\n        except Exception as error:\n            pending.put(("error", error))\n        finally:\n            pending.put(("end", None))\n    reader = threading.Thread(target=read_output, daemon=True)\n    reader.start()\n    try:\n        while True:\n            try:\n                kind, value = pending.get(timeout=min(0.5, HEARTBEAT_SECONDS))\n            except queue.Empty:\n                kind, value = "wait", None\n            if kind == "end":\n                break\n            if kind == "error":\n                raise value\n            if kind == "line":\n                last_output = time.monotonic()\n                tail.append(value)\n                log.write(value)\n                log.flush()\n                print(value, end="", flush=True)\n            now = time.monotonic()\n            if now - last_tick >= HEARTBEAT_SECONDS and process.poll() is None:\n                message = (f"[ĐANG CHẠY] {step} · PID {process.pid} · {now-started:.0f}s"\n                           f" · tổng đã chạy {time.perf_counter()-total_started:.1f}s"\n                           f" · chưa có log mới {now-last_output:.0f}s (không phải % hoàn thành)")\n                print(message, flush=True)\n                log.write(message + "\\n")\n                log.flush()\n                last_tick = now\n        code = process.wait()\n    except BaseException:\n        process.terminate()\n        try:\n            process.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        raise\n    finally:\n        reader.join(timeout=1)\n        process.stdout.close()\n    if code:\n        raise RuntimeError(f"{step} lỗi, mã {code}.\\n" + "".join(tail)[-10000:])\n\n\ndef child_environment(dependencies):\n    return dict(os.environ, PYTHONPATH=str(dependencies), PYTHONUNBUFFERED="1", PYTHONUTF8="1",\n                TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")\n\n\ndef ensure_dependencies(root, log):\n    key = hashlib.sha256(json.dumps([list(sys.version_info[:2]), REQUIREMENTS]).encode()).hexdigest()[:12]\n    parent = Path(root) / "dependencies"\n    target = parent / key\n    # Chỉ tái sử dụng THƯ VIỆN đã cài, không nhập code/UI/pipeline cũ.\n    for candidate in (target, Path("/kaggle/working/hybrid_ocr_clean/dependencies") / key):\n        try:\n            if json.loads((candidate / "ready.json").read_text(encoding="utf-8"))["requirements"] == REQUIREMENTS:\n                print("Dùng lại thư viện OCR riêng:", candidate, flush=True)\n                return candidate\n        except (OSError, ValueError, KeyError, TypeError):\n            pass\n    parent.mkdir(parents=True, exist_ok=True)\n    stage = Path(tempfile.mkdtemp(prefix=key + "-", dir=parent))\n    stream_command([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(stage),\n                    "--progress-bar", "off", "--disable-pip-version-check", "--no-warn-conflicts",\n                    "--no-warn-script-location", *REQUIREMENTS], log=log, step="Cài thư viện riêng lần đầu")\n    check = \'\'\'\nimport importlib.metadata as md\nimport torch, torchvision, transformers, tokenizers, PIL, numpy, pypandoc\nfrom transformers import AutoModel, AutoTokenizer, StoppingCriteria, StoppingCriteriaList\nfrom packaging.requirements import Requirement\nassert transformers.__version__ == "4.46.3"\nassert tokenizers.__version__ == "0.20.3"\nfor package in ("transformers", "tokenizers", "huggingface-hub", "accelerate", "requests", "Pillow", "pypandoc_binary"):\n    for raw in md.requires(package) or []:\n        requirement = Requirement(raw)\n        if requirement.marker and not requirement.marker.evaluate({"extra": ""}):\n            continue\n        if requirement.specifier and md.version(requirement.name) not in requirement.specifier:\n            raise RuntimeError(package + ": " + raw)\nprint("Dependency check OK | torch=" + torch.__version__ + " | transformers=" + transformers.__version__)\n\'\'\'\n    stream_command([sys.executable, "-c", check], log=log, env=child_environment(stage), step="Kiểm tra thư viện")\n    (stage / "ready.json").write_text(json.dumps({"requirements": REQUIREMENTS}), encoding="utf-8")\n    if not target.exists():\n        stage.rename(target)\n        return target\n    return stage\n\n\ndef image_path(ref, work_dir):\n    value = unquote(ref.strip().strip("<>").replace(r"\\ ", " "))\n    if urlsplit(value).scheme and not re.match(r"^[A-Za-z]:[/\\\\]", value):\n        return None\n    path = Path(value)\n    resolved = (path if path.is_absolute() else Path(work_dir) / path).resolve()\n    return resolved if resolved.is_relative_to(Path(work_dir).resolve()) else None\n\n\ndef load_rgb(path):\n    from PIL import Image\n    with Image.open(path) as source:\n        rgba = source.convert("RGBA")\n        canvas = Image.new("RGBA", rgba.size, "white")\n        canvas.alpha_composite(rgba)\n        return canvas.convert("RGB")\n\n\ndef is_coordinate_plot(image):\n    """Nhận diện bảo thủ: lưới xám đều + một/vài trục đậm bên trong hình.\n\n    Không coi mọi bảng có đường kẻ là đồ thị. Đây vẫn là heuristic; không phải\n    mô hình hiểu/giải đồ thị và không loại được mọi dạng hình tọa độ.\n    """\n    import numpy as np\n    from PIL import Image\n    if min(image.size) < 70 or not 0.45 <= image.width / image.height <= 2.5:\n        return False\n    if max(image.size) > 1000:\n        scale = 1000 / max(image.size)\n        image = image.resize(tuple(max(1, round(value * scale)) for value in image.size), Image.Resampling.LANCZOS)\n    array = np.asarray(image.convert("L"))\n    def centres(mask, axis):\n        indices = np.flatnonzero(mask.mean(axis=axis) > 0.6)\n        if not len(indices):\n            return []\n        groups = np.split(indices, np.flatnonzero(np.diff(indices) > 1) + 1)\n        return [float(group.mean()) for group in groups]\n    def regular(points):\n        if len(points) < 5:\n            return False\n        gaps = np.diff(points)\n        spacing = float(np.median(gaps))\n        return spacing >= 5 and float(((gaps > spacing * 0.7) & (gaps < spacing * 1.3)).mean()) >= 0.75\n    light_rows, light_cols = centres(array < 240, 1), centres(array < 240, 0)\n    dark_rows, dark_cols = centres(array < 180, 1), centres(array < 180, 0)\n    height, width = array.shape\n    return (regular(light_rows) and regular(light_cols)\n            and 1 <= len(dark_rows) <= 3 and 1 <= len(dark_cols) <= 3\n            and len(light_rows) >= len(dark_rows) + 3 and len(light_cols) >= len(dark_cols) + 3\n            and any(0.1 * height < point < 0.9 * height for point in dark_rows)\n            and any(0.1 * width < point < 0.9 * width for point in dark_cols))\n\n\ndef analyze_image_for_ocr(image):\n    """Heuristic gốc, thêm trường hợp lưới tọa độ bị nhận nhầm là mixed."""\n    import numpy as np\n    if is_coordinate_plot(image):\n        return "coordinate_plot"\n    colors = image.resize((64, 64)).getcolors(64 * 64)\n    if (len(colors) if colors else 9999) > 400:\n        return "diagram"\n    array = np.asarray(image.convert("L"), dtype=np.float32)\n    height, width = array.shape\n    if min(height, width) < 2:\n        return "mixed"\n    dark_ratio = (array < 128).mean()\n    edge_density = (np.abs(np.diff(array, axis=0)) > 30).mean() + (np.abs(np.diff(array, axis=1)) > 30).mean()\n    bh, bw = max(1, height // 8), max(1, width // 8)\n    variances = [array[i:i+bh, j:j+bw].std() for i in range(0, height-bh, bh) for j in range(0, width-bw, bw)]\n    spread = np.std(variances) if variances else 0\n    is_text = 0.03 < dark_ratio < 0.6 and edge_density > 0.08\n    if dark_ratio < 0.15 and edge_density < 0.12 and spread > 20 and not is_text:\n        return "diagram"\n    return "text" if is_text else "mixed"\n\n\ndef preprocess_for_ocr(image, destination):\n    """Crop/mốc 1200 px như gốc; không ép mọi ảnh lên 384 px."""\n    import numpy as np\n    from PIL import Image, ImageOps\n    width, height = image.size\n    mask = np.asarray(image.convert("L")) < 240\n    rows, cols = np.where(mask.any(axis=1))[0], np.where(mask.any(axis=0))[0]\n    if len(rows) and len(cols):\n        box = (max(0, int(cols[0])-8), max(0, int(rows[0])-8),\n               min(width, int(cols[-1])+9), min(height, int(rows[-1])+9))\n        if (box[2]-box[0]) * (box[3]-box[1]) < width * height * 0.85:\n            image = image.crop(box)\n    if max(image.size) > 1200:\n        scale = 1200 / max(image.size)\n        image = image.resize(tuple(max(1, round(size*scale)) for size in image.size), Image.Resampling.LANCZOS)\n    if np.asarray(image.convert("L")).std() < 40:\n        image = ImageOps.autocontrast(image, cutoff=2)\n    image.save(destination, format="PNG", optimize=False)\n    return image.size\n\n\ndef plan_images(markdown, work_dir, *, ocr_diagrams=False, ocr_small_images=True):\n    records, by_ref, seen = [], {}, {}\n    prepared = Path(work_dir) / "preprocessed"\n    prepared.mkdir(exist_ok=True)\n    for match in IMAGE_RE.finditer(markdown):\n        ref = match[1]\n        if ref in by_ref:\n            continue\n        record = dict(ref=ref, status="pending", ocr_text="", raw_ocr="", attempts=[])\n        records.append(record)\n        by_ref[ref] = record\n        path = image_path(ref, work_dir)\n        if path is None or not path.is_file():\n            record.update(status="missing", error="Không tìm được ảnh Pandoc tham chiếu.")\n            continue\n        try:\n            image = load_rgb(path)\n        except Exception as error:\n            record.update(status="unsupported", error=str(error))\n            continue\n        record["source_size"] = list(image.size)\n        key = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()\n        record["key"] = key\n        if key in seen:\n            record.update(status="duplicate", duplicate_of=seen[key]["ref"])\n            continue\n        seen[key] = record\n        if image.convert("L").point(lambda pixel: 255 if pixel < 250 else 0).getbbox() is None:\n            record["status"] = "blank"\n            continue\n        if not ocr_small_images and min(image.size) < 80:\n            record["status"] = "small_skipped"\n            continue\n        label = analyze_image_for_ocr(image)\n        record["classification"] = label\n        record["render_policy"] = "reference_only" if label == "coordinate_plot" else "inline"\n        if label in {"diagram", "coordinate_plot"} and not ocr_diagrams:\n            record["status"] = "diagram_kept"\n            continue\n        target = prepared / (key[:20] + ".png")\n        record["prepared_size"] = list(preprocess_for_ocr(image, target))\n        record.update(status="queued", prepared_path=str(target))\n    return records, by_ref\n\n\ndef clean_ocr(text):\n    """Chỉ lọc mẫu rác rõ ràng; không tuyên bố chuỗi còn lại là đúng."""\n    value = text.strip() if text else ""\n    if not value or any(re.search(pattern, value) for pattern in GARBAGE_PATTERNS):\n        return ""\n    if value.count("| 100 |") > 5 or ("Point" in value and value.count("| 10 |") > 5):\n        return ""\n    if "Category" in value and "Value" in value:\n        return ""\n    tokens = re.split(r"[\\s|<>/]+", value)\n    if sum(bool(re.fullmatch(r"[&#$%^*`]+\\d{0,2}\\*{0,2}", token)) for token in tokens if len(token) <= 4) >= 4:\n        return ""\n    if "<tr" in value.lower():\n        rows = []\n        for row in re.findall(r"<tr\\b[^>]*>(.*?)</tr>", value, re.DOTALL | re.IGNORECASE):\n            cells = re.findall(r"<t[dh]\\b[^>]*>(.*?)</t[dh]>", row, re.DOTALL | re.IGNORECASE)\n            if cells:\n                rows.append([html.unescape(re.sub(r"<[^>]+>", "", cell)).strip().replace("|", r"\\|").replace("\\n", " ") for cell in cells])\n        if rows:\n            columns = max(map(len, rows))\n            padded = [row + [""] * (columns - len(row)) for row in rows]\n            lines = ["| " + " | ".join(row) + " |" for row in padded]\n            lines.insert(1, "| " + " | ".join(["---"] * columns) + " |")\n            return "\\n".join(lines)\n    # Không bỏ x, 0, π; không sửa LaTeX, ma trận hay dấu *.\n    return value\n\n\ndef clean_pandoc_output(text):\n    # Không chạy các regex tự đoán ngoặc, xóa dấu nhân, bỏ cột ma trận của bản dán.\n    return text.strip()\n\n\ndef is_plot_label_list(text):\n    """Chỉ nhận danh sách nhãn rời x/y/O + số, không nhận biểu thức/toán LaTeX."""\n    value = (text or "").strip().replace("−", "-").replace("–", "-")\n    tokens = value.split()\n    if len(tokens) < 6:\n        return False\n    lowered = {token.lower() for token in tokens}\n    if not {"x", "y"}.issubset(lowered):\n        return False\n    numeric = re.compile(r"[+-]?(?:\\d+(?:[.,]\\d+)?|[.]\\d+)")\n    return (sum(bool(numeric.fullmatch(token)) for token in tokens) >= 3\n            and all(token.lower() in {"x", "y", "o"} or numeric.fullmatch(token) for token in tokens))\n\n\ndef remove_axis_label_blocks(page):\n    """Dọn bản HTML đã có: chỉ bỏ block [OCR] chứa danh sách nhãn trục rời.\n\n    Không gọi model, không bỏ hình, không sửa công thức hoặc blockquote khác.\n    """\n    class BlockFilter(HTMLParser):\n        def __init__(self):\n            super().__init__(convert_charrefs=False)\n            self.output, self.buffer, self.words = [], [], []\n            self.depth = self.removed = 0\n        def append(self, value):\n            (self.buffer if self.depth else self.output).append(value)\n        def handle_starttag(self, tag, attrs):\n            if tag == "blockquote":\n                if not self.depth:\n                    self.buffer, self.words = [], []\n                self.depth += 1\n            self.append(self.get_starttag_text())\n        def handle_startendtag(self, tag, attrs):\n            self.append(self.get_starttag_text())\n        def handle_endtag(self, tag):\n            self.append("</" + tag + ">")\n            if self.depth and tag == "blockquote":\n                self.depth -= 1\n                if not self.depth:\n                    text = " ".join(self.words).strip()\n                    header = re.match(r"^\\[OCR\\b[^\\]]*\\]\\s*", text, re.IGNORECASE)\n                    if header and is_plot_label_list(text[header.end():]):\n                        self.removed += 1\n                    else:\n                        self.output.extend(self.buffer)\n                    self.buffer, self.words = [], []\n        def handle_data(self, data):\n            self.append(data)\n            if self.depth:\n                self.words.append(data)\n        def handle_entityref(self, name):\n            self.append("&" + name + ";")\n            if self.depth:\n                self.words.append(html.unescape("&" + name + ";"))\n        def handle_charref(self, name):\n            self.append("&#" + name + ";")\n            if self.depth:\n                self.words.append(html.unescape("&#" + name + ";"))\n        def handle_comment(self, data):\n            self.append("<!--" + data + "-->")\n        def handle_decl(self, decl):\n            self.append("<!" + decl + ">")\n        def handle_pi(self, data):\n            self.append("<?" + data + ">")\n    parser = BlockFilter()\n    parser.feed(page)\n    parser.close()\n    if parser.buffer:\n        parser.output.extend(parser.buffer)\n    return "".join(parser.output), parser.removed\n\n\ndef ocr_configs(size):\n    longest = max(size)\n    sides = [384] if longest <= 130 else ([448, 512, 640] if longest <= 350 else [512, 640, 448])\n    return [dict(crop_mode=False, base_size=side, image_size=side) for side in sides]\n\n\ndef make_stopper(tokenizer, base_class):\n    class GarbageStoppingCriteria(base_class):\n        def __init__(self):\n            self.step = 0\n        def __call__(self, input_ids, scores, **kwargs):\n            self.step += 1\n            if self.step % 15 or input_ids.shape[1] < 80:\n                return False\n            text = tokenizer.decode(input_ids[0][-80:], skip_special_tokens=True)\n            tokens = re.split(r"[\\s|<>/]+", text)\n            return sum(bool(re.fullmatch(r"[&#$%^*`]+\\d{0,2}\\*{0,2}", token)) for token in tokens if token and len(token) <= 4) >= 4\n    return GarbageStoppingCriteria()\n\n\ndef infer_image(model, tokenizer, record, out_dir, torch):\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for index, config in enumerate(ocr_configs(record["prepared_size"])):\n        started = time.perf_counter()\n        attempt_dir = out_dir / f"a{index}"\n        attempt = dict(config=config, status="pending")\n        record["attempts"].append(attempt)\n        print(f"    Lượt {index+1}: {config[\'image_size\']}px", flush=True)\n        try:\n            with torch.inference_mode():\n                model.infer(tokenizer, prompt="<image>\\nFree OCR.",\n                            image_file=record["prepared_path"], output_path=str(attempt_dir),\n                            save_results=True, test_compress=False, **config)\n            output = attempt_dir / "result.mmd"\n            raw = output.read_text(encoding="utf-8").strip() if output.is_file() else ""\n            record["raw_ocr"] = raw\n            cleaned = clean_ocr(raw)\n            attempt.update(status="text_returned" if cleaned else "empty_or_filtered", characters=len(raw))\n            if cleaned:\n                record.update(status="ocr_text_returned", ocr_text=cleaned)\n                if is_plot_label_list(cleaned):\n                    record.update(render_policy="reference_only", annotation_warning="Chỉ có nhãn trục/số rời; không chèn thành công thức trong đề.")\n                return\n        except torch.cuda.OutOfMemoryError as error:\n            attempt.update(status="out_of_memory", error=str(error))\n            torch.cuda.empty_cache()\n        except Exception as error:\n            attempt.update(status="error", error=str(error))\n            print("    Lỗi lượt OCR:", error, flush=True)\n        finally:\n            attempt["seconds"] = round(time.perf_counter() - started, 3)\n    record["status"] = "ocr_unread"\n\n\ndef build_final(markdown, by_ref):\n    def replace(match):\n        record = by_ref.get(match[1], {})\n        text = record.get("ocr_text", "")\n        if not text or record.get("render_policy") == "reference_only" or is_plot_label_list(text):\n            return match[0]  # Luôn giữ hình gốc, không thay bằng OCR thất bại.\n        block = "\\n".join("> " + line if line else ">" for line in text.splitlines())\n        # A blank line is required before a pipe table. Another blank line\n        # after the block keeps following source text outside the blockquote.\n        return match[0] + "\\n\\n> **[OCR — cần đối chiếu]**\\n>\\n" + block + "\\n\\n"\n    return IMAGE_RE.sub(replace, clean_pandoc_output(markdown))\n\n\ndef write_reports(records, metrics, output):\n    portable = json.loads(json.dumps(records, ensure_ascii=False))\n    for record in portable:\n        if record.get("prepared_path"):\n            record["prepared_path"] = Path(record["prepared_path"]).relative_to(Path(output)).as_posix()\n    (Path(output) / "image-report.json").write_text(json.dumps(portable, ensure_ascii=False, indent=2), encoding="utf-8")\n    (Path(output) / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef ast_text(value, skip_math=False):\n    if isinstance(value, dict):\n        kind = value.get("t")\n        if kind == "Str":\n            return value.get("c", "")\n        if kind in {"Space", "SoftBreak", "LineBreak"}:\n            return " "\n        if kind == "Math":\n            return "" if skip_math else value["c"][1]\n        return ast_text(value.get("c", []), skip_math)\n    if isinstance(value, list):\n        return "".join(ast_text(item, skip_math) for item in value)\n    return ""\n\n\ndef has_ast_type(value, kind):\n    if isinstance(value, dict):\n        return value.get("t") == kind or has_ast_type(value.get("c", []), kind)\n    return isinstance(value, list) and any(has_ast_type(item, kind) for item in value)\n\n\ndef prepare_print_ast(document):\n    """Reconstruct only parsed tables/math; preserve original images for review.\n\n    No values, symbols or equations are inferred or repaired here.\n    """\n    counts = dict(structured_tables=0, inline_math=0, notes_only=0)\n    def wrap(kind, classes, content):\n        return {"t": kind, "c": [["", classes, []], content]}\n    def ocr_parts(block):\n        if not isinstance(block, dict) or block.get("t") != "BlockQuote":\n            return None\n        blocks = block.get("c", [])\n        if not blocks or blocks[0].get("t") not in {"Para", "Plain"}:\n            return None\n        inlines = blocks[0]["c"]\n        if not inlines or inlines[0].get("t") != "Strong":\n            return None\n        if not re.fullmatch(r"\\[OCR\\b[^\\]]*\\]", ast_text(inlines[0]).strip(), re.IGNORECASE):\n            return None\n        remainder = inlines[1:]\n        while remainder and remainder[0].get("t") in {"Space", "SoftBreak"}:\n            remainder = remainder[1:]\n        return ([{"t": "Para", "c": remainder}] if remainder else []) + blocks[1:]\n    def process(items):\n        output = []\n        for block in items:\n            parts = ocr_parts(block)\n            previous = output[-1] if output else None\n            if parts is None or not isinstance(previous, dict) or previous.get("t") not in {"Para", "Plain"}:\n                output.append(block)\n                continue\n            images = [index for index, inline in enumerate(previous["c"]) if inline.get("t") == "Image"]\n            if not images:\n                output.append(block)\n                continue\n            is_table = has_ast_type(parts, "Table")\n            is_math = (len(parts) == 1 and parts[0].get("t") in {"Para", "Plain"}\n                       and has_ast_type(parts, "Math")\n                       and not re.sub(r"[\\s.,;:()[\\]{}…-]", "", ast_text(parts, skip_math=True)))\n            if not is_table and not is_math:\n                output.append(wrap("Div", ["ocr-notes"], parts))\n                counts["notes_only"] += 1\n                continue\n            index = images[-1]\n            before, image, after = previous["c"][:index], previous["c"][index], previous["c"][index+1:]\n            output.pop()\n            if is_table:\n                rendered = [wrap("Div", ["ocr-caption"], [part])\n                            if part.get("t") in {"Para", "Plain"}\n                            and re.fullmatch(r"(?:Bảng|Table)\\s+\\d+[.:]?", ast_text(part).strip(), re.IGNORECASE)\n                            else part for part in parts]\n                if ast_text(before).strip() or has_ast_type(before, "Math") or has_ast_type(before, "Image"):\n                    output.append({"t": previous["t"], "c": before})\n                output.append(wrap("Div", ["ocr-pair", "ocr-table"], [\n                    wrap("Div", ["ocr-source"], [{"t": "Para", "c": [image]}]),\n                    wrap("Div", ["ocr-rendered"], rendered),\n                ]))\n                if ast_text(after).strip() or has_ast_type(after, "Math") or has_ast_type(after, "Image"):\n                    output.append({"t": previous["t"], "c": after})\n                counts["structured_tables"] += 1\n            else:\n                pair = wrap("Span", ["ocr-pair", "ocr-math"], [\n                    wrap("Span", ["ocr-source"], [image]),\n                    wrap("Span", ["ocr-rendered"], parts[0]["c"]),\n                ])\n                output.append({"t": previous["t"], "c": [*before, pair, *after]})\n                counts["inline_math"] += 1\n        return output\n    def walk(value):\n        if isinstance(value, dict):\n            return {key: walk(item) for key, item in value.items()}\n        if isinstance(value, list):\n            return process([walk(item) for item in value])\n        return value\n    return walk(deepcopy(document)), counts\n\n\ndef page_css(source):\n    namespace = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"\n    values = dict(w=210, h=297, top=15, right=15, bottom=15, left=15)\n    with zipfile.ZipFile(source) as archive:\n        document = ET.fromstring(archive.read("word/document.xml"))\n        sections = list(document.iter(namespace + "sectPr"))\n        if sections:\n            for name, keys in (("pgSz", ("w", "h")), ("pgMar", ("top", "right", "bottom", "left"))):\n                node = sections[-1].find(namespace + name)\n                if node is not None:\n                    for key in keys:\n                        try:\n                            values[key] = float(node.get(namespace + key)) * 25.4 / 1440\n                        except (ValueError, TypeError):\n                            pass\n    width = max(50, values["w"] - values["left"] - values["right"])\n    return (f"@page{{size:{values[\'w\']}mm {values[\'h\']}mm;margin:{values[\'top\']}mm {values[\'right\']}mm {values[\'bottom\']}mm {values[\'left\']}mm}}"\n            f"body{{font:12pt/1.3 \'Times New Roman\',serif;max-width:{width}mm;margin:20px auto;background:white}}"\n            "*{box-sizing:border-box}img{max-width:100%}p{margin:0 0 6pt;orphans:3;widows:3}"\n            "table{border-collapse:collapse;max-width:100%;margin:6pt auto;break-inside:auto}"\n            "td,th{border:0.6pt solid #222;padding:5pt 7pt;vertical-align:middle;overflow-wrap:anywhere}"\n            "th{text-align:center;font-weight:bold}thead{display:table-header-group}tr{break-inside:avoid}"\n            "math{font-size:1em}blockquote{margin:6pt 0;padding:0 6pt}"\n            ".ocr-source,.ocr-notes{display:none}"\n            ".show-source .ocr-source,.show-source .ocr-notes{display:block}"\n            ".show-source .ocr-math .ocr-source{display:inline-block;vertical-align:middle;margin-right:6pt}"\n            ".show-source .ocr-source,.ocr-notes{border:1px dashed #aaa;padding:5pt;margin:6pt 0;background:#fff9da}"\n            ".ocr-pair{max-width:100%}.ocr-rendered{max-width:100%}.ocr-rendered>p:last-child{margin-bottom:0}"\n            ".ocr-caption{text-align:center;font-style:italic;margin:3pt 0 8pt}"\n            ".toolbar{font:14px Arial;border-bottom:1px solid #bbb;padding:10px;margin-bottom:12px}"\n            "@media print{body{max-width:none;margin:0;color:#000;background:#fff}"\n            ".toolbar{display:none!important}.ocr-source{display:none!important}.ocr-notes{display:none!important}"\n            "img{break-inside:avoid}table{break-inside:auto}tr{break-inside:avoid}"\n            "thead{display:table-header-group}a{color:inherit;text-decoration:none}}")\n\n\ndef export_html(final_path, source, pandoc, *, media_root=None, title=None):\n    work_dir = Path(final_path).parent\n    asset_root = Path(media_root) if media_root is not None else work_dir\n    parsed = subprocess.run([pandoc, str(final_path), "-f", "markdown+tex_math_dollars+tex_math_single_backslash-raw_html", "-t", "json"],\n                            cwd=asset_root, capture_output=True, text=True, encoding="utf-8", timeout=120)\n    if parsed.returncode:\n        raise RuntimeError("Không phân tích được kết quả OCR: " + parsed.stderr)\n    document, counts = prepare_print_ast(json.loads(parsed.stdout))\n    process = subprocess.run([pandoc, "-f", "json", "-t", "html5", "--mathml", "--wrap=none"],\n                             input=json.dumps(document), cwd=asset_root, capture_output=True, text=True, encoding="utf-8", timeout=120)\n    if process.returncode:\n        raise RuntimeError("Không xuất được HTML kết quả OCR: " + process.stderr)\n    warnings = (parsed.stderr + "\\n" + process.stderr).strip()\n    if warnings:\n        (work_dir / "html-warnings.log").write_text(warnings, encoding="utf-8")\n        print("[CẢNH BÁO XUẤT HTML]", warnings, flush=True)\n    def embed(match):\n        ref = html.unescape(match[1])\n        path = image_path(ref, asset_root)\n        if path is None or not path.is_file():\n            return match[0]\n        suffix = path.suffix.lower()\n        if suffix not in {".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp"}:\n            return \'<span style="color:#a00">[Ảnh \' + html.escape(path.name) + \' chưa hiển thị được trong trình duyệt; xem file gốc trong ZIP]</span>\'\n        mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".svg": "image/svg+xml"}.get(suffix, "image/" + suffix[1:])\n        uri = "data:" + mime + ";base64," + base64.b64encode(path.read_bytes()).decode("ascii")\n        return match[0].replace(match[1], uri, 1)\n    body = re.sub(r\'<img\\b[^>]*\\bsrc="([^"]+)"[^>]*>\', embed, process.stdout)\n    body, _ = remove_axis_label_blocks(body)\n    display_name = title or Path(source).name\n    toolbar = (\'<div class="toolbar"><strong>Đề đang xem: \' + html.escape(display_name) + \'</strong><br>\'\n               \'<button onclick="window.print()">In / lưu PDF kết quả</button>\'\n               \'<label><input type="checkbox" onchange="document.body.classList.toggle(\\\'show-source\\\',this.checked)"> Đối chiếu ảnh gốc / ghi chú OCR</label>\'\n               \'<p>Bản in dùng bảng/công thức đã dựng từ OCR khi có cấu trúc; đồ thị và ảnh chưa dựng được giữ nguyên. \'\n               \'Chữ OCR chưa được xác nhận đúng. Bật đối chiếu để kiểm tra; ảnh đối chiếu và ghi chú không in kèm. \'\n               \'In tỷ lệ 100%, tắt Headers/Footers của trình duyệt. Ngắt dòng/ngắt trang có thể khác Word.</p></div>\')\n    page = (\'<!doctype html><html lang="vi"><meta charset="utf-8"><title>Kết quả OCR — \'\n            + html.escape(display_name) + \'</title><style>\' + page_css(source)\n            + \'</style><body>\' + toolbar + body + \'</body></html>\')\n    target = work_dir / "ket-qua-ocr.html"\n    target.write_text(page, encoding="utf-8")\n    (work_dir / "print-layout.json").write_text(json.dumps(dict(**counts, policy="structured_ocr_reconstruction_with_source_comparison",\n        validation="layout_only_not_ocr_accuracy", media_root=str(asset_root)), ensure_ascii=False, indent=2), encoding="utf-8")\n    return target\n\n\ndef run_worker(source, output, *, ocr_diagrams=False, ocr_small_images=True):\n    import pypandoc\n    source, output = validate_docx(source), Path(output).resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(source, output / "input.docx")\n    started = time.perf_counter()\n    pandoc = pypandoc.get_pandoc_path()\n    step1 = output / "step1.md"\n    print("\\nBUOC 1: Bóc DOCX bằng Pandoc", flush=True)\n    process = subprocess.run([pandoc, "-f", "docx", str(source), "-t", "markdown+tex_math_dollars", "--wrap=none",\n                              "--extract-media=media", "-o", str(step1)], cwd=output,\n                             capture_output=True, text=True, encoding="utf-8", timeout=120)\n    if process.returncode:\n        raise RuntimeError("Pandoc lỗi: " + process.stderr)\n    if process.stderr.strip():\n        (output / "pandoc-warnings.log").write_text(process.stderr, encoding="utf-8")\n        print("[CẢNH BÁO PANDOC]", process.stderr, flush=True)\n    markdown = step1.read_text(encoding="utf-8")\n    records, by_ref = plan_images(markdown, output, ocr_diagrams=ocr_diagrams, ocr_small_images=ocr_small_images)\n    tasks = [record for record in records if record["status"] == "queued"]\n    metrics = dict(model=MODEL_NAME, dtype="bfloat16", attention="eager", parse_preprocess_s=time.perf_counter()-started,\n                   model_load_s=0.0, ocr_s=0.0, gpu_inference_ran=False, images_referenced=len(records),\n                   queued_unique_images=len(tasks), ocr_text_returned=0, ocr_diagrams=ocr_diagrams,\n                   native_math_policy="unchanged_pandoc_latex", unsupported_images=sum(record["status"] == "unsupported" for record in records))\n    with zipfile.ZipFile(source) as archive:\n        metrics["source_media_files"] = [name for name in archive.namelist() if name.startswith("word/media/") and not name.endswith("/")]\n        document = ET.fromstring(archive.read("word/document.xml"))\n        metrics["native_word_equations_main_doc"] = sum(1 for node in document.iter("{http://schemas.openxmlformats.org/officeDocument/2006/math}oMath"))\n    print("Kiểm tra ảnh Pandoc xuất được. DOCX gốc được giữ trong ZIP; bản này không chuyển WMF/EMF/OLE bằng LibreOffice.", flush=True)\n    print("Word Equation trong văn bản gốc:", metrics["native_word_equations_main_doc"], "— do Pandoc đọc trực tiếp.", flush=True)\n    print(f"Ảnh tham chiếu: {len(records)} · ảnh cần OCR: {len(tasks)} · sơ đồ giữ nguyên: "\n          f"{sum(record[\'status\'] == \'diagram_kept\' for record in records)} · ảnh không đọc được: {metrics[\'unsupported_images\']}", flush=True)\n    for record in records:\n        if record["status"] in {"unsupported", "missing", "small_skipped"}:\n            print("[CHƯA OCR]", record["ref"], record.get("error", record["status"]), flush=True)\n    write_reports(records, metrics, output)\n    if tasks:\n        import gc\n        import torch\n        from transformers import AutoModel, AutoTokenizer, StoppingCriteria, StoppingCriteriaList\n        if not torch.cuda.is_available():\n            raise RuntimeError("Ảnh cần OCR nhưng không có GPU CUDA. Bật GPU trong Settings.")\n        free, _ = torch.cuda.mem_get_info(0)\n        if free < 7_000_000_000:\n            raise RuntimeError("GPU còn dưới 7 GB VRAM. Dừng model khác trước khi chạy.")\n        print("\\nBUOC 2: Nạp", MODEL_NAME, "| GPU:", torch.cuda.get_device_name(0), flush=True)\n        load_start = time.perf_counter()\n        gc.collect()\n        torch.cuda.empty_cache()\n        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)\n        model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16,\n                                         attn_implementation="eager", low_cpu_mem_usage=True).eval().cuda()\n        original_generate = model.generate\n        stopper = make_stopper(tokenizer, StoppingCriteria)\n        def capped(self, *args, **kwargs):\n            stopper.step = 0\n            kwargs["max_new_tokens"] = MAX_NEW_TOKENS\n            kwargs["eos_token_id"] = tokenizer.eos_token_id\n            kwargs["pad_token_id"] = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id\n            existing = list(kwargs.get("stopping_criteria") or [])\n            kwargs["stopping_criteria"] = StoppingCriteriaList([*existing, stopper])\n            return original_generate(*args, **kwargs)\n        model.generate = types.MethodType(capped, model)\n        metrics["model_load_s"] = time.perf_counter() - load_start\n        print("\\nBUOC 3: OCR ảnh đã chọn", flush=True)\n        ocr_start = time.perf_counter()\n        try:\n            for index, record in enumerate(tasks, 1):\n                print(f"[{index}/{len(tasks)}] {record[\'ref\']}", flush=True)\n                metrics["gpu_inference_ran"] = True\n                infer_image(model, tokenizer, record, output / "ocr_imgs" / f"img_{index:03d}", torch)\n                metrics["ocr_text_returned"] += bool(record["ocr_text"])\n                print("    " + ("Có chữ trả về — cần đối chiếu" if record["ocr_text"] else "CHƯA ĐỌC ĐƯỢC"), flush=True)\n                write_reports(records, metrics, output)\n        finally:\n            metrics["ocr_s"] = time.perf_counter() - ocr_start\n            del original_generate, model\n            gc.collect()\n            torch.cuda.empty_cache()\n    else:\n        print("Không có ảnh được chọn để OCR. Không nạp/chạy model; không phải benchmark OCR.", flush=True)\n    for record in records:\n        if record["status"] == "duplicate":\n            original = by_ref[record["duplicate_of"]]\n            record.update(ocr_text=original["ocr_text"], raw_ocr=original["raw_ocr"],\n                          render_policy=original.get("render_policy", "inline"))\n    print("\\nBUOC 4: Ghép và hiển thị nội dung trích xuất + OCR", flush=True)\n    final = build_final(markdown, by_ref)\n    final_path = output / "final.md"\n    final_path.write_text(final, encoding="utf-8")\n    export_start = time.perf_counter()\n    export_html(final_path, source, pandoc)\n    metrics.update(export_s=time.perf_counter()-export_start, pipeline_s=time.perf_counter()-started,\n                   images_unread=sum(record["status"] in {"unsupported", "missing", "ocr_unread", "small_skipped"} for record in records),\n                   inference_attempts=sum(len(record["attempts"]) for record in records),\n                   coordinate_plots=sum(record.get("classification") == "coordinate_plot" for record in records),\n                   ocr_reference_only=sum(bool(record["ocr_text"]) and (record.get("render_policy") == "reference_only" or is_plot_label_list(record["ocr_text"])) for record in records))\n    write_reports(records, metrics, output)\n    print(f"\\nPandoc/ảnh: {metrics[\'parse_preprocess_s\']:.2f}s · nạp model: {metrics[\'model_load_s\']:.2f}s"\n          f" · OCR: {metrics[\'ocr_s\']:.2f}s · xuất HTML: {metrics[\'export_s\']:.2f}s", flush=True)\n    print(f"Ảnh có chữ OCR: {metrics[\'ocr_text_returned\']}/{len(tasks)} — KHÔNG phải tỷ lệ chính xác.", flush=True)\n    if metrics["ocr_reference_only"]:\n        print(f"{metrics[\'ocr_reference_only\']} kết quả đồ thị/nhãn trục chỉ lưu để đối chiếu trong image-report.json, không chèn vào đề.", flush=True)\n    print("\\n===== NỘI DUNG KẾT QUẢ =====\\n" + final, flush=True)\n    return dict(results=str(output), final=str(final_path), html=str(output / "ket-qua-ocr.html"), metrics=metrics)\n\n\ndef self_script_path(root):\n    """Hỗ trợ cả chạy file .py và copy toàn bộ file vào một cell notebook."""\n    filename = globals().get("__file__")\n    if filename and Path(filename).is_file():\n        content = Path(filename).read_text(encoding="utf-8")\n        if "def self_script_path(" in content and "def run_worker(" in content:\n            return Path(filename).resolve()\n    from IPython import get_ipython\n    shell = get_ipython()\n    history = getattr(getattr(shell, "history_manager", None), "input_hist_raw", [])\n    source = history[-1] if history else ""\n    if "def run_worker(" not in source or "def self_script_path(" not in source:\n        raise RuntimeError("Hãy copy TOÀN BỘ file Python vào cùng một cell, hoặc dùng notebook đi kèm.")\n    compile(source, "copied-original-fixed-cell", "exec")\n    target = Path(root) / "hybrid_ocr_original_fixed.py"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    target.write_text(source, encoding="utf-8")\n    return target.resolve()\n\n\ndef print_timing(metrics):\n    """Hiện lại số đo đã lưu, không chạy lại model để xem thời gian."""\n    total = metrics.get("total_s")\n    note = ""\n    if total is None:\n        total = metrics.get("total_before_zip_s")\n        note = " — bản cũ: chưa gồm đóng ZIP"\n    if total is None:\n        return\n    minutes, seconds = divmod(round(float(total), 2), 60)\n    print(f"\\nTỔNG THỜI GIAN ĐÃ CHẠY: {int(minutes)} phút {seconds:.2f} giây ({total:.2f}s){note}", flush=True)\n    labels = (("dependency_setup_s", "Chuẩn bị/thư viện"), ("parse_preprocess_s", "Pandoc/ảnh"),\n              ("model_load_s", "Nạp model"), ("ocr_s", "OCR"),\n              ("export_s", "Xuất HTML"), ("zip_s", "Đóng ZIP"))\n    parts = [f"{label}: {metrics[key]:.2f}s" for key, label in labels if key in metrics]\n    if parts:\n        print(" · ".join(parts), flush=True)\n\n\ndef display_result(result):\n    """Hiện đầu ra thật, dùng được ngay khi copy script vào notebook."""\n    from IPython.display import HTML, FileLink, IFrame, display\n    for path, mime, label in (\n        (result["html"], "text/html", "Tải HTML kết quả OCR — mở để In / lưu PDF"),\n        (result["final"], "text/markdown", "Tải Markdown kết quả"),\n        (result["archive"], "application/zip", "Tải ZIP (kết quả + ảnh + DOCX gốc + báo cáo + log)"),\n    ):\n        path = Path(path)\n        if path.stat().st_size <= 10_000_000:\n            encoded = base64.b64encode(path.read_bytes()).decode("ascii")\n            display(HTML(\'<p><a download="\' + html.escape(path.name) + \'" href="data:\' + mime + \';base64,\' + encoded + \'">\' + label + \'</a></p>\'))\n        else:\n            display(FileLink(str(path), result_html_prefix=label + ": "))\n            print("File lớn: có thể tải ở mục Output của Kaggle:", path)\n    page = Path(result["html"]).read_text(encoding="utf-8")\n    display(IFrame("about:blank", width="100%", height=1100, extras=[\n        \'title="Kết quả trích xuất và OCR"\', \'sandbox="allow-scripts allow-modals allow-downloads"\',\n        \'srcdoc="\' + html.escape(page, quote=True) + \'"\',\n    ]))\n    print_timing(result.get("metrics", {}))\n    print("Kết quả được lưu tại:", result["results"])\n\n\ndef run(source=INPUT_DOCX, root=RUN_ROOT, *, ocr_diagrams=OCR_DIAGRAMS, ocr_small_images=OCR_SMALL_IMAGES):\n    """Chạy trong tiến trình riêng; không thay thư viện của kernel notebook."""\n    started = time.perf_counter()\n    source = validate_docx(source)  # Chặn đường dẫn rỗng/thư mục trước khi cài gì.\n    root = Path(root).resolve()\n    job = root / "runs" / (datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid4().hex[:8])\n    job.mkdir(parents=True)\n    log_path, output = job / "run.log", job / "result"\n    print("File đề:", source, "\\nLog:", log_path, flush=True)\n    script = self_script_path(root)\n    try:\n        with log_path.open("w", encoding="utf-8") as log:\n            dependencies = ensure_dependencies(root, log)\n            setup_s = time.perf_counter() - started\n            command = [sys.executable, "-u", str(script), "--worker", "--input", str(source), "--output", str(output)]\n            if ocr_diagrams:\n                command.append("--ocr-diagrams")\n            if not ocr_small_images:\n                command.append("--skip-small-images")\n            stream_command(command, log=log, env=child_environment(dependencies), step="Pandoc + DeepSeek-OCR gốc", run_started=started)\n        metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))\n        metrics.update(source_path=str(source), source_name=source.name,\n                       source_sha256=hashlib.sha256(source.read_bytes()).hexdigest())\n        metrics["dependency_setup_s"] = setup_s\n        metrics["total_before_zip_s"] = time.perf_counter() - started\n        shutil.copyfile(log_path, output / "run.log")\n        archive = job / "ket-qua-ocr.zip"\n        zip_started = time.perf_counter()\n        # Close the compressed payload before reading the final timer. Append\n        # the tiny timing report once afterwards, so ZIP/filesystem/UI agree\n        # without duplicate metrics entries or recompressing all images.\n        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:\n            for path in sorted(output.rglob("*")):\n                if path.is_file() and path != output / "metrics.json":\n                    zipped.write(path, path.relative_to(output).as_posix())\n        metrics["zip_s"] = time.perf_counter() - zip_started\n        metrics["total_s"] = time.perf_counter() - started\n        metrics["timing_scope"] = "run entry through ZIP payload close; excludes timing-report finalization and screen rendering"\n        metrics_path = output / "metrics.json"\n        metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\n        with zipfile.ZipFile(archive, "a", compression=zipfile.ZIP_DEFLATED) as zipped:\n            zipped.write(metrics_path, "metrics.json")\n        print("\\nĐÃ XONG. Kết quả:", output, "\\nZIP:", archive, flush=True)\n        print_timing(metrics)\n        return dict(results=output, archive=archive, final=output / "final.md", html=output / "ket-qua-ocr.html", metrics=metrics,\n                    source_path=str(source))\n    except BaseException:\n        print("\\nLượt chạy chưa hoàn tất. Giữ nguyên dữ liệu và log tại:", log_path, flush=True)\n        print(f"Thời gian đến lúc dừng/lỗi: {time.perf_counter() - started:.2f}s", flush=True)\n        raise\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input", default=INPUT_DOCX)\n    parser.add_argument("--root", default=RUN_ROOT)\n    parser.add_argument("--worker", action="store_true")\n    parser.add_argument("--output")\n    parser.add_argument("--ocr-diagrams", action="store_true")\n    parser.add_argument("--skip-small-images", action="store_true")\n    args = parser.parse_args(argv)\n    if args.worker:\n        if not args.output:\n            parser.error("--worker cần --output")\n        return run_worker(args.input, args.output, ocr_diagrams=args.ocr_diagrams, ocr_small_images=not args.skip_small_images)\n    return run(args.input, args.root, ocr_diagrams=args.ocr_diagrams, ocr_small_images=not args.skip_small_images)\n\n\nif __name__ == "__main__":\n    try:\n        from IPython import get_ipython\n        running_in_notebook = get_ipython() is not None\n    except ImportError:\n        running_in_notebook = False\n    if running_in_notebook:\n        result = None\n        FINAL_PATH = WORK_DIR = CLEAN_HTML = None\n        result = run(INPUT_DOCX, RUN_ROOT, ocr_diagrams=OCR_DIAGRAMS, ocr_small_images=OCR_SMALL_IMAGES)\n        FINAL_PATH, WORK_DIR = result["final"], result["results"]\n        display_result(result)\n    else:\n        main()\n'
root = Path(RUN_ROOT)
root.mkdir(parents=True, exist_ok=True)
script_path = root / "hybrid_ocr_original_fixed.py"
script_path.write_text(SCRIPT_SOURCE, encoding="utf-8")
spec = importlib.util.spec_from_file_location("hybrid_original_fixed_run", script_path)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
result = None
FINAL_PATH = WORK_DIR = CLEAN_HTML = None
PICKER_SOURCE = '"""Copy toàn bộ vào một cell mới: chọn DOCX thật, rồi bấm Đọc đề đã chọn.\n\nChỉ là bộ chọn file cho pipeline đã có, không thay model/cấu hình OCR.\nKhông tự chọn đề Toán. Không khởi chạy OCR khi chỉ mở giao diện.\n"""\nimport base64\nimport builtins\nimport hashlib\nimport html\nimport importlib.util\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport traceback\n\ntry:\n    import ipywidgets as widgets\nexcept ModuleNotFoundError:\n    subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets>=8,<9"], check=True)\n    import ipywidgets as widgets\nfrom IPython.display import HTML, FileLink, IFrame, display\n\n\ndef load_existing_pipeline(namespace):\n    module = namespace.get("pipeline")\n    if callable(getattr(module, "run", None)):\n        return module\n    root = Path(namespace.get("RUN_ROOT", "/kaggle/working/hybrid_original_fixed"))\n    path = root / "hybrid_ocr_original_fixed.py"\n    if not path.is_file():\n        raise RuntimeError("Chưa tìm thấy pipeline đã nạp. Cần cell OCR chính đã được thiết lập trong cùng phiên.")\n    source = path.read_text(encoding="utf-8")\n    if "def run_worker(" not in source or "def self_script_path(" not in source:\n        raise RuntimeError("File pipeline không đúng bản Pandoc + DeepSeek-OCR đang dùng.")\n    spec = importlib.util.spec_from_file_location("existing_exam_pipeline", path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    namespace["pipeline"] = module\n    return module\n\n\nclass ExamPicker:\n    def __init__(self, namespace, runner=None, search_root="/kaggle/input"):\n        self.namespace = namespace\n        self.runner = runner if runner is not None else load_existing_pipeline(namespace)\n        self.search_root = Path(search_root)\n        self.busy = self.refreshing = False\n        self.choice = widgets.Dropdown(description="Chọn DOCX:", options=[("— Chọn đề muốn đọc —", "")],\n                                       layout=widgets.Layout(width="98%"))\n        self.refresh_button = widgets.Button(description="Làm mới danh sách", icon="refresh")\n        self.run_button = widgets.Button(description="Đọc đề đã chọn", icon="play", button_style="success", disabled=True,\n                                         layout=widgets.Layout(width="240px"))\n        self.status = widgets.HTML("Chọn đúng tên file ở trên, sau đó bấm Đọc đề đã chọn.")\n        self.logs = widgets.Output(layout=widgets.Layout(max_height="350px", overflow="auto"))\n        self.preview = widgets.Output()\n        self.choice.observe(self.on_choose, names="value")\n        self.refresh_button.on_click(self.refresh)\n        self.run_button.on_click(self.on_run)\n        self.view = widgets.VBox([widgets.HTML("<h3>Đọc một đề khác</h3><p>Chọn file cụ thể bên dưới. Không tự lấy đề Toán hay kết quả cũ.</p>"),\n                                  self.choice, widgets.HBox([self.refresh_button, self.run_button]), self.status, self.logs, self.preview])\n        self.refresh()\n\n    def clear_saved_result(self):\n        # Clear in-memory references only. Old files remain on disk.\n        for name in ("result", "FINAL_PATH", "WORK_DIR", "CLEAN_HTML"):\n            self.namespace[name] = None\n        self.preview.clear_output()\n\n    def refresh(self, _=None):\n        if self.busy:\n            return\n        previous = self.choice.value\n        files = sorted(path.resolve() for path in self.search_root.rglob("*")\n                       if path.is_file() and path.suffix.lower() == ".docx") if self.search_root.is_dir() else []\n        self.refreshing = True\n        try:\n            self.choice.options = [("— Chọn đề muốn đọc —", "")] + [(path.name + " — " + str(path.parent), str(path)) for path in files]\n            self.choice.value = previous if previous in {str(path) for path in files} else ""\n        finally:\n            self.refreshing = False\n        self.run_button.disabled = not bool(self.choice.value)\n        if not self.choice.value:\n            self.clear_saved_result()\n            self.namespace["INPUT_DOCX"] = ""\n            self.status.value = (f"Tìm thấy {len(files)} DOCX. <b>Chọn đề muốn đọc</b>; chưa chạy OCR." if files else\n                                 "Chưa có DOCX trong Input. Thêm file vào Input rồi bấm Làm mới danh sách.")\n\n    def on_choose(self, change):\n        if self.busy or self.refreshing:\n            return\n        self.clear_saved_result()\n        self.namespace["INPUT_DOCX"] = change["new"]\n        self.run_button.disabled = not bool(change["new"])\n        if change["new"]:\n            self.status.value = "<b>Đã chọn, chưa chạy:</b> " + html.escape(Path(change["new"]).name)\n        else:\n            self.status.value = "Chưa chọn đề."\n\n    def log_print(self, *args, sep=" ", end="\\n", file=None, flush=False):\n        if file is not None:\n            return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)\n        self.logs.append_stdout(sep.join(map(str, args)) + end)\n\n    def render_result(self, result, source):\n        self.preview.append_display_data(HTML("<h3>Kết quả của: " + html.escape(source.name) + "</h3>"))\n        for key, mime, label in (("html", "text/html", "Tải HTML để xem / in"),\n                                 ("final", "text/markdown", "Tải Markdown"), ("archive", "application/zip", "Tải ZIP")):\n            path = Path(result[key])\n            if path.stat().st_size <= 10_000_000:\n                encoded = base64.b64encode(path.read_bytes()).decode("ascii")\n                self.preview.append_display_data(HTML(\'<p><a download="\' + html.escape(path.name) + \'" href="data:\' + mime + \';base64,\' + encoded + \'">\' + label + \'</a></p>\'))\n            else:\n                self.preview.append_display_data(FileLink(str(path), result_html_prefix=label + ": "))\n        page = Path(result["html"]).read_text(encoding="utf-8")\n        self.preview.append_display_data(IFrame("about:blank", width="100%", height=1000, extras=[\n            \'title="Kết quả của \' + html.escape(source.name, quote=True) + \'"\',\n            \'sandbox="allow-scripts allow-modals allow-downloads"\', \'srcdoc="\' + html.escape(page, quote=True) + \'"\']))\n        timing = result.get("metrics", {})\n        seconds = timing.get("total_s", timing.get("total_before_zip_s"))\n        if seconds is not None:\n            self.preview.append_stdout(f"Tổng thời gian: {seconds:.2f}s ({seconds/60:.2f} phút)\\n")\n\n    def on_run(self, _=None):\n        if self.busy or not self.choice.value:\n            return\n        source = Path(self.choice.value).resolve()\n        self.clear_saved_result()\n        self.logs.clear_output()\n        self.namespace["INPUT_DOCX"] = str(source)\n        self.busy = True\n        for control in (self.choice, self.refresh_button, self.run_button):\n            control.disabled = True\n        self.run_button.description = "Đang đọc đề đã chọn…"\n        self.status.value = "<b>ĐANG ĐỌC:</b> " + html.escape(source.name) + "<br>" + html.escape(str(source))\n        missing = object()\n        old_print = getattr(self.runner, "print", missing)\n        self.runner.print = self.log_print\n        started = time.monotonic()\n        try:\n            source = self.runner.validate_docx(source)\n            result = self.runner.run(source, self.namespace.get("RUN_ROOT", "/kaggle/working/hybrid_original_fixed"),\n                                     ocr_diagrams=bool(self.namespace.get("OCR_DIAGRAMS", False)),\n                                     ocr_small_images=bool(self.namespace.get("OCR_SMALL_IMAGES", True)))\n            if not isinstance(result, dict) or not result.get("results") or not result.get("html"):\n                raise RuntimeError("Lượt chạy chưa trả về kết quả hoàn chỉnh.")\n            copied_input = Path(result["results"]) / "input.docx"\n            if not copied_input.is_file() or hashlib.sha256(copied_input.read_bytes()).digest() != hashlib.sha256(source.read_bytes()).digest():\n                raise RuntimeError("Không hiển thị: file DOCX của kết quả không khớp đề đã chọn. Đã chặn kết quả cũ.")\n            self.render_result(result, source)\n            result["source_path"] = str(source)\n            self.namespace.update(result=result, FINAL_PATH=result["final"], WORK_DIR=result["results"])\n            self.status.value = "<b>ĐÃ XONG:</b> " + html.escape(source.name) + f" · {time.monotonic()-started:.1f}s. Kết quả mới ở bên dưới."\n        except (Exception, KeyboardInterrupt) as error:\n            self.clear_saved_result()\n            self.status.value = "<b>CHƯA ĐỌC ĐƯỢC:</b> " + html.escape(source.name) + "<br>" + html.escape(str(error))\n            self.logs.append_stderr(traceback.format_exc())\n        finally:\n            if old_print is missing:\n                delattr(self.runner, "print")\n            else:\n                self.runner.print = old_print\n            self.busy = False\n            self.choice.disabled = self.refresh_button.disabled = False\n            self.run_button.disabled = not bool(self.choice.value)\n            self.run_button.description = "Đọc đề đã chọn"\n\n\ndef show_exam_picker(namespace, runner=None, search_root="/kaggle/input"):\n    picker = ExamPicker(namespace, runner, search_root)\n    display(picker.view)\n    return picker\n\n\nif __name__ == "__main__":\n    exam_picker = show_exam_picker(globals())\n'
picker_namespace = {"__name__": "exam_file_picker"}
exec(compile(PICKER_SOURCE, "<exam-file-picker>", "exec"), picker_namespace)
exam_picker = picker_namespace["show_exam_picker"](globals(), pipeline)
